In [1]:
from sympy import *
init_printing()

In [2]:
import pandas as pd

# Load the dataset
file_path = "../Data_odds.xlsx"  # Adjust path if needed
df = pd.read_excel(file_path)

# Create match outcome target variable (1X2) and rename to "result"
df["result"] = df.apply(lambda row: "1" if row["Goals_home_team"] > row["goals_away_team"] 
                                  else ("X" if row["Goals_home_team"] == row["goals_away_team"] 
                                        else "2"), axis=1)

# Add home/away indicator (1 = home team, 0 = away team)
df["home_away_flag"] = 1  # Since all listed teams are home teams

# Convert match outcome to numerical points
df["match_points_home"] = df["result"].map({"1": 3, "X": 1, "2": 0})
df["match_points_away"] = df["result"].map({"1": 0, "X": 1, "2": 3})

# Ensure matches are sorted by date before rolling calculations
df = df.sort_values(by=["date"])

# Compute PPG for Last 5 Games (rolling mean) for Home and Away
df["PPG_home_last_5"] = df.groupby("home_team_name")["match_points_home"].transform(lambda x: x.rolling(5, min_periods=1).mean())
df["PPG_away_last_5"] = df.groupby("away_team_name.x")["match_points_away"].transform(lambda x: x.rolling(5, min_periods=1).mean())

# **FIXED**: Compute overall PPG (Home + Away Combined)
ppg_all_matches = df[["home_team_name", "match_points_home", "date"]].rename(columns={"home_team_name": "team", "match_points_home": "points"})
ppg_away_matches = df[["away_team_name.x", "match_points_away", "date"]].rename(columns={"away_team_name.x": "team", "match_points_away": "points"})

# Merge all matches (home + away) into one dataset for rolling calculations
ppg_combined = pd.concat([ppg_all_matches, ppg_away_matches]).sort_values(by=["team", "date"])

# Compute rolling PPG for last 5 matches (home + away combined)
ppg_combined["PPG_last_5"] = ppg_combined.groupby("team")["points"].transform(lambda x: x.rolling(5, min_periods=1).mean())

# Merge back into original dataframe
df = df.merge(ppg_combined[["team", "date", "PPG_last_5"]], left_on=["home_team_name", "date"], right_on=["team", "date"], how="left")
df.drop(columns=["team"], inplace=True)  # Remove extra column

# Avoid division by zero by replacing zero shots on target with a small value
df["home_team_shots_on_target"] = df["home_team_shots_on_target"].replace(0, 1e-6)
df["away_team_shots_on_target"] = df["away_team_shots_on_target"].replace(0, 1e-6)

# Compute Conversion Rate (Goals / Shots on Target)
df["conversion_rate_home"] = df["Goals_home_team"] / df["home_team_shots_on_target"]
df["conversion_rate_away"] = df["goals_away_team"] / df["away_team_shots_on_target"]

# Define predictor variables (without odds)
predictor_variables = [
    "date", "home_team_name", "away_team_name.x", "league", "home_away_flag",
    "PPG_home_last_5", "PPG_away_last_5", "PPG_last_5",
    "average_goals_per_match_pre_match", "average_corners_per_match_pre_match",
    "average_cards_per_match_pre_match",
    "home_team_shots", "away_team_shots",
    "home_team_shots_on_target", "away_team_shots_on_target",
    "home_team_fouls", "away_team_fouls",
    "home_team_possession", "away_team_possession",
    "home_team_yellow_cards", "away_team_yellow_cards",
    "home_team_red_cards", "away_team_red_cards",
    "Corners_home_team", "Corners_away_team",
    "Passes_home_team", "Passes_away_team",
    "accurate_passes_home_team", "Passes.accurate_away_team",
    "xG_home_team", "xG_away_team",
    "PPDA_home_team", "PPDA_away_team",
    "Match.tempo_home_team", "Match.tempo_away_team",
    "Average.pass.length_home_team", "Average.pass.length_away_team",
    "Shots.outside.PA_home_team", "Shots.outside.PA.on_target_home_team",
    "Positional.attacks_home_team", "Counterattacks_home_team",
    "Defensive.duels_won_home_team", "Defensive.duels.won_away_team",
    "conversion_rate_home", "conversion_rate_away"
]

# Create final dataset for predictions
df_filtered = df[["result"] + predictor_variables]

# Create a separate DataFrame for odds
odds_columns = ["date", "home_team_name", "away_team_name.x",
                "odds_ft_home_team_win", "odds_ft_draw", "odds_ft_away_team_win"]
df_odds = df[odds_columns]

# Check for missing values
missing_values = df_filtered.isnull().sum()
missing_values

# Save the processed datasets
df_filtered.to_csv("football_data_clean.csv", index=True)
df_odds.to_csv("odds_data.csv", index=True)


In [3]:
df_filtered.head()

,result,date,home_team_name,away_team_name.x,league,home_away_flag,PPG_home_last_5,PPG_away_last_5,PPG_last_5,average_goals_per_match_pre_match,...,Average.pass.length_home_team,Average.pass.length_away_team,Shots.outside.PA_home_team,Shots.outside.PA.on_target_home_team,Positional.attacks_home_team,Counterattacks_home_team,Defensive.duels_won_home_team,Defensive.duels.won_away_team,conversion_rate_home,conversion_rate_away
0,2,2015-07-24,MSV Duisburg,Kaiserslautern,2. Bundesliga,1,0.0,3.0,0.0,0.0,...,24.32,21.95,7,3,28,0,37,22,0.250000,1.50
1,2,2015-07-25,FSV Frankfurt,RB Leipzig,2. Bundesliga,1,0.0,3.0,0.0,0.0,...,23.30,20.45,3,0,23,8,65,68,0.000000,0.50
2,1,2015-07-25,Greuther Fürth,Karlsruher SC,2. Bundesliga,1,3.0,0.0,3.0,0.0,...,19.84,19.82,5,1,30,3,46,49,0.333333,0.00
3,X,2015-07-25,St. Pauli,Arminia Bielefeld,2. Bundesliga,1,1.0,1.0,1.0,0.0,...,22.54,23.58,9,2,31,0,25,34,0.000000,0.00
4,2,2015-07-26,Eintracht Braunschweig,Sandhausen,2. Bundesliga,1,0.0,3.0,0.0,0.0,...,21.74,21.78,6,3,41,2,42,71,0.166667,0.75
